<a href="https://colab.research.google.com/github/szymonbloch/tuberculosis_detection/blob/main/3_layer_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import glob
import os
import numpy as np
import joblib
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

# ==========================================
# 1. WCZYTANIE ETYKIET Z PLIKÓW TXT
# ==========================================
POS_TXT_PATH = '/content/drive/MyDrive/NTWI_project/data/positive.txt'
NEG_TXT_PATH = '/content/drive/MyDrive/NTWI_project/data/negative.txt'

with open(POS_TXT_PATH, 'r') as f:
    pos_slides = [line.strip() for line in f if line.strip()]

with open(NEG_TXT_PATH, 'r') as f:
    neg_slides = [line.strip() for line in f if line.strip()]

# Budujemy słownik mapujący: {nazwa_slajdu: 0 lub 1}
labels_dict = {}
for slide in pos_slides:
    labels_dict[slide] = 1
for slide in neg_slides:
    labels_dict[slide] = 0

df_final_3l = pd.DataFrame(list(labels_dict.items()), columns=['image_name', 'image_output'])


In [ ]:
# ==========================================
# 2. DETEKTOR ANOMALII (Połączenie 3 warstw za pomocą MAX)
# ==========================================
ANOMALY_3L_PATH = '/content/drive/MyDrive/NTWI_project/data/anomaly_detector/3-layer_ad_cnn_04_06_2025_00_31_21/*.csv'
anomaly_files = glob.glob(ANOMALY_3L_PATH)
table_list_anomaly = []

for file in anomaly_files:
    basename = os.path.basename(file)
    if '_Wholeslide_Default_' not in basename:
        continue

    # Wyciągamy nazwę pacjenta oraz warstwę z nazwy pliku
    slide_name = basename.split('_Wholeslide_Default_')[0]
    layer = basename.split('_Wholeslide_Default_')[1].replace('.csv', '')

    # Ignorujemy warstwę Extended, bo chcemy sami zasymulować pooling 3D z warstw -0.8, 0, 0.8
    if layer == 'Extended':
        continue

    df_temp = pd.read_csv(file)
    df_temp['probability'] = df_temp['probability'].astype(str).str.extract(r'tensor\(([0-9.]+)').astype(float)

    # Wyciągamy współrzędne kafelka (np. 147_208)
    df_temp['tile_coord'] = df_temp['image_name'].astype(str).str.split('.').str[0].str.strip()

    # Tworzymy unikalny, globalny klucz kafelka
    df_temp['tile_name'] = slide_name + '_' + df_temp['tile_coord']
    df_temp['image_name'] = slide_name

    df_temp = df_temp[['tile_name', 'image_name', 'probability', 'output']]
    table_list_anomaly.append(df_temp)

df_anomaly_3l_raw = pd.concat(table_list_anomaly, ignore_index=True)

# Spłaszczanie w pionie: bierzemy maksimum z 3 warstw ostrości dla danego punktu XY pacjenta
df_anomaly_3l = df_anomaly_3l_raw.groupby(['tile_name', 'image_name']).agg({
    'probability': 'max',
    'output': 'max'
}).reset_index()
df_anomaly_3l.rename(columns={'output': 'tile_output'}, inplace=True)


In [ ]:
# ==========================================
# 3. CECHY KOLORYSTYCZNE (bez continue + poprawka NaN)
# ==========================================
COLORS_3L_PATH = '/content/drive/MyDrive/NTWI_project/data/color_features/3_layers/*.csv'
color_files = glob.glob(COLORS_3L_PATH)
table_list_colors = []

for file in color_files:
    basename = os.path.basename(file)

    if '_Wholeslide_Default_' in basename:
        slide_name = basename.split('_Wholeslide_Default_')[0]

        df_temp = pd.read_csv(file, header=None, sep=None, engine='python')
        df_temp.rename(columns={0: 'tile_coord'}, inplace=True)
        df_temp['tile_coord'] = df_temp['tile_coord'].astype(str).str.split('.').str[0].str.strip()
        #tworzenie unikalnego klucza
        df_temp['tile_name'] = slide_name + '_' + df_temp['tile_coord']
        df_temp.drop(columns=['tile_coord'], inplace=True)

        table_list_colors.append(df_temp)

df_colors_3l_raw = pd.concat(table_list_colors, ignore_index=True)

# USUWANIE ZŁOŚLIWEJ KOLUMNY 59
df_colors_3l_raw = df_colors_3l_raw.dropna(axis=1, how='all')
if 59 in df_colors_3l_raw.columns:
    df_colors_3l_raw = df_colors_3l_raw.drop(columns=[59])
if '59' in df_colors_3l_raw.columns:
    df_colors_3l_raw = df_colors_3l_raw.drop(columns=['59'])

# Uśredniamy kolory
df_colors_3l = df_colors_3l_raw.groupby('tile_name').mean().reset_index()


In [ ]:
# ==========================================
# 4. WIELKI MERGE DANYCH 3D
# ==========================================
df_step1_3l = df_anomaly_3l.merge(df_final_3l, on='image_name', how='inner')
df_merged_3l = df_step1_3l.merge(df_colors_3l, on='tile_name', how='inner')



In [ ]:
# ==========================================
# 5. AGREGACJA TOP-50 DO POZIOMU PACJENTA
# ==========================================
K = 50
top_k_bags_3l = []
labels_3l = []
slide_names_3l = []

grouped = df_merged_3l.groupby('image_name')

for name, group in grouped:
    group_sorted = group.sort_values(by='probability', ascending=False)
    top_k_tiles = group_sorted.head(K)

    label = top_k_tiles['image_output'].iloc[0]
    labels_3l.append(label)
    slide_names_3l.append(name)

    features_cols = ['probability'] + [col for col in group.columns if str(col).isdigit()]
    flat_features = top_k_tiles[features_cols].values.flatten()

    if len(flat_features) < K * len(features_cols):
        padding = np.zeros(K * len(features_cols) - len(flat_features))
        flat_features = np.concatenate((flat_features, padding))

    top_k_bags_3l.append(flat_features)

X_test_3l = np.array(top_k_bags_3l)
y_test_3l = np.array(labels_3l)

In [6]:
# ==========================================
# 6. POBRANIE WYTRENOWANEGO MODELU I TEST
# ==========================================
# Wczytujemy model, który zapisałeś na dysku pod koniec Etapu 1
rf_model_loaded = joblib.load('/content/drive/MyDrive/NTWI_project/models/best_k5_rf.pkl')

# Sprawdzamy jak zamrożony model radzi sobie na nowych danych 3D
y_pred_3l = rf_model_loaded.predict(X_test_3l)
bacc_3l = balanced_accuracy_score(y_test_3l, y_pred_3l)

print(f"\nOstateczny wynik Balanced Accuracy na danych 3-warstwowych: {bacc_3l * 100:.2f}%")
print("Macierz pomyłek dla danych 3D:\n", confusion_matrix(y_test_3l, y_pred_3l))

ValueError: X has 3000 features, but RandomForestClassifier is expecting 2950 features as input.